# Equality Constrained Optimization: Minimum Distance to Target Blend

This notebook demonstrates GMM and Flow Matching approaches for solving equality-constrained optimization problems.

**Authors:** Victor Alves and John R. Kitchin

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import warnings
import os

# Force CPU
os.environ['JAX_PLATFORMS'] = 'cpu'

import torch

# Import reusable utilities
from generative_optimization import (
    generate_samples,
    best_gmm,
    ConditionalFlowMatching
)

# Figure settings for publication
mpl.rcParams['figure.facecolor'] = 'white'
mpl.rcParams['axes.facecolor'] = 'white'
mpl.rcParams['figure.dpi'] = 150
mpl.rcParams['font.size'] = 11
mpl.rcParams['axes.labelsize'] = 12
mpl.rcParams['axes.titlesize'] = 12
mpl.rcParams['legend.fontsize'] = 10

warnings.filterwarnings('ignore')

# Set seeds
torch.manual_seed(42)
np.random.seed(42)

print("Setup complete")

---
## Equality Constrained Optimization: Minimum Distance to Target Blend

A refinery wants to find the blend composition closest to a target recipe while meeting an octane specification. This is a **quadratic programming** problem with an interior solution.

**Objective:** Minimize deviation from target blend $(x_1^*, x_2^*, x_3^*) = (0.4, 0.4, 0.2)$
$$\min_{x_1, x_2, x_3} \quad (x_1 - 0.4)^2 + (x_2 - 0.4)^2 + (x_3 - 0.2)^2$$

**Constraints:**
1. Meet target octane (RON = 87): $95 x_1 + 82 x_2 + 94 x_3 = 87$
2. Mass balance: $x_1 + x_2 + x_3 = 1$

| Component | Target | RON |
|-----------|--------|-----|
| Reformate ($x_1$) | 0.40 | 95 |
| Isomerate ($x_2$) | 0.40 | 82 |
| Alkylate ($x_3$)  | 0.20 | 94 |

The Lagrangian:
$$\mathcal{L} = (x_1-0.4)^2 + (x_2-0.4)^2 + (x_3-0.2)^2 + \lambda_1(95x_1 + 82x_2 + 94x_3 - 87) + \lambda_2(x_1 + x_2 + x_3 - 1)$$

The KKT conditions give 5 equations in 5 unknowns $(x_1, x_2, x_3, \lambda_1, \lambda_2)$.

In [ ]:
# Minimum distance blending problem - quadratic objective with interior solution
import jax
import jax.numpy as jnp

# Target blend and RON values
x_target = np.array([0.4, 0.4, 0.2])
rons = np.array([95.0, 82.0, 94.0])
target_ron = 87.0

# Lagrangian function
@jax.jit
def lagrangian(vars):
    """Lagrangian for minimum distance blending problem."""
    x1, x2, x3, lam1, lam2 = vars
    # Quadratic objective: sum of squared deviations from target
    obj = (x1 - 0.4)**2 + (x2 - 0.4)**2 + (x3 - 0.2)**2
    # Constraints
    g1 = 95*x1 + 82*x2 + 94*x3 - 87  # Octane constraint
    g2 = x1 + x2 + x3 - 1             # Mass balance
    return obj + lam1*g1 + lam2*g2

grad_lagrangian = jax.jit(jax.grad(lagrangian))

# Scipy reference solution
from scipy.optimize import minimize

def objective(x):
    return np.sum((x - x_target)**2)

result = minimize(
    objective,
    x0=[0.33, 0.34, 0.33],
    method='SLSQP',
    constraints=[
        {'type': 'eq', 'fun': lambda x: rons @ x - target_ron},
        {'type': 'eq', 'fun': lambda x: np.sum(x) - 1}
    ]
)

x_scipy = result.x
print("Scipy Reference Solution:")
print(f"  x1 (Reformate) = {x_scipy[0]:.6f}")
print(f"  x2 (Isomerate) = {x_scipy[1]:.6f}")
print(f"  x3 (Alkylate)  = {x_scipy[2]:.6f}")
print(f"  Objective      = {result.fun:.6f}")
print(f"  Blend RON      = {rons @ x_scipy:.2f}")
print(f"  Sum            = {x_scipy.sum():.6f}")

# Verify this is an interior solution (all components > 0)
print(f"\nAll components positive: {all(x_scipy > 0.01)}")

### Train GMM for Blending Optimization

In [ ]:
# Generate training data: sample variables and compute gradients
n_blend_samples = 10000

# Sample blend fractions and Lagrange multipliers
# λ2 ≈ -4.5 at the solution, so we sample a range that includes it
x1_samples = np.random.uniform(0, 1, n_blend_samples)
x2_samples = np.random.uniform(0, 1, n_blend_samples)
x3_samples = np.random.uniform(0, 1, n_blend_samples)
lam1_samples = np.random.uniform(-0.5, 0.5, n_blend_samples)
lam2_samples = np.random.uniform(-6, 0, n_blend_samples)

# Compute gradients for each sample
gradients = []
for i in range(n_blend_samples):
    vars_i = jnp.array([x1_samples[i], x2_samples[i], x3_samples[i], 
                        lam1_samples[i], lam2_samples[i]])
    grad_i = grad_lagrangian(vars_i)
    gradients.append(np.array(grad_i))

gradients = np.array(gradients)

# Build joint dataset: [x1, x2, x3, λ1, λ2, ∂L/∂x1, ∂L/∂x2, ∂L/∂x3, ∂L/∂λ1, ∂L/∂λ2]
data_blend = np.column_stack([
    x1_samples, x2_samples, x3_samples,
    lam1_samples, lam2_samples,
    gradients
])

print(f"Training data shape: {data_blend.shape}")
print(f"Columns: [x1, x2, x3, λ1, λ2, ∂L/∂x1, ∂L/∂x2, ∂L/∂x3, ∂L/∂λ1, ∂L/∂λ2]")

# Train GMM
gmm_blend, gmm_blend_info = best_gmm(data_blend, verbose=True)
print(f"\nBest GMM: {gmm_blend_info['best_k']} components")

In [ ]:
# GMM: Condition on all gradients = 0 (KKT conditions)
condition_indices = [5, 6, 7, 8, 9]  # Gradient columns
condition_values = [[0.0, 0.0, 0.0, 0.0, 0.0]]

c_gmm_blend = gmm_blend.condition(condition_indices, condition_values)
gmm_blend_samples = c_gmm_blend.sample(1000)

# Extract blend fractions
x1_gmm = gmm_blend_samples[:, 0]
x2_gmm = gmm_blend_samples[:, 1]
x3_gmm = gmm_blend_samples[:, 2]

print("GMM Solution (conditioning on ∇L = 0):")
print(f"  x1 (Reformate) = {x1_gmm.mean():.4f} ± {x1_gmm.std():.4f}")
print(f"  x2 (Isomerate) = {x2_gmm.mean():.4f} ± {x2_gmm.std():.4f}")
print(f"  x3 (Alkylate)  = {x3_gmm.mean():.4f} ± {x3_gmm.std():.4f}")

### Train Flow Matching for Blending Optimization

In [ ]:
# Train Flow Matching model for blending problem
# x_data: variables to generate [x1, x2, x3, λ1, λ2]
# c_data: conditions (gradients) [∂L/∂x1, ∂L/∂x2, ∂L/∂x3, ∂L/∂λ1, ∂L/∂λ2]

x_data_blend = data_blend[:, 0:5]  # Variables
c_data_blend = data_blend[:, 5:10]  # Gradients

fm_blend = ConditionalFlowMatching(x_dim=5, c_dim=5, hidden_dim=128, n_layers=4)
fm_blend_history = fm_blend.fit(x_data_blend, c_data_blend, epochs=3000, batch_size=128, verbose=True)

In [ ]:
# FM: Sample conditioned on all gradients = 0
fm_blend_samples = fm_blend.sample(c_values=[[0.0, 0.0, 0.0, 0.0, 0.0]], n_samples=1000, n_steps=200)

# Extract blend fractions
x1_fm = fm_blend_samples[:, 0]
x2_fm = fm_blend_samples[:, 1]
x3_fm = fm_blend_samples[:, 2]

print("Flow Matching Solution (conditioning on ∇L = 0):")
print(f"  x1 (Reformate) = {x1_fm.mean():.4f} ± {x1_fm.std():.4f}")
print(f"  x2 (Isomerate) = {x2_fm.mean():.4f} ± {x2_fm.std():.4f}")
print(f"  x3 (Alkylate)  = {x3_fm.mean():.4f} ± {x3_fm.std():.4f}")

### Publication Figure: GMM vs Flow Matching for Blending

In [ ]:
# Create publication figure: 3-panel comparison of blend fractions
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

component_names = ['$x_1$ (Reformate)', '$x_2$ (Isomerate)', '$x_3$ (Alkylate)']
gmm_data = [x1_gmm, x2_gmm, x3_gmm]
fm_data = [x1_fm, x2_fm, x3_fm]
scipy_vals = x_scipy
target_vals = x_target

for i, ax in enumerate(axes):
    bins = np.linspace(0, 1, 50)
    
    # Overlapping histograms
    ax.hist(gmm_data[i], bins=bins, alpha=0.5, color='red', 
            label='GMM', density=True)
    ax.hist(fm_data[i], bins=bins, alpha=0.5, color='blue', 
            label='Flow Matching', density=True)
    
    # Scipy reference solution
    ax.axvline(scipy_vals[i], color='black', ls='-', lw=2, label='Scipy')
    # Target blend
    ax.axvline(target_vals[i], color='green', ls='--', lw=2, label='Target')
    
    ax.set_xlabel(component_names[i])
    ax.set_ylabel('Density' if i == 0 else '')
    ax.set_title(f'({chr(97+i)}) {component_names[i]}')
    ax.grid(True, alpha=0.3)
    ax.set_xlim(0, 1)
    
    if i == 0:
        ax.legend(loc='upper right', fontsize=9)

plt.suptitle('Minimum Distance Blending: GMM vs Flow Matching', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('blending_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nFigure saved as 'blending_comparison.png'")

### Quantitative Comparison

In [ ]:
print("Minimum Distance Blending Optimization Comparison")
print("=" * 70)
print(f"Objective: Minimize distance to target blend while achieving RON = {target_ron}")
print(f"Target blend: x1={x_target[0]}, x2={x_target[1]}, x3={x_target[2]}")
print()

print(f"{'Method':<20} {'x1':<12} {'x2':<12} {'x3':<12} {'Objective':<12}")
print("-" * 60)

# Scipy reference
obj_scipy = np.sum((x_scipy - x_target)**2)
print(f"{'Scipy (SLSQP)':<20} {x_scipy[0]:<12.4f} {x_scipy[1]:<12.4f} {x_scipy[2]:<12.4f} {obj_scipy:<12.6f}")

# GMM
x_gmm_mean = np.array([x1_gmm.mean(), x2_gmm.mean(), x3_gmm.mean()])
obj_gmm = np.sum((x_gmm_mean - x_target)**2)
print(f"{'GMM':<20} {x_gmm_mean[0]:<12.4f} {x_gmm_mean[1]:<12.4f} {x_gmm_mean[2]:<12.4f} {obj_gmm:<12.6f}")

# Flow Matching
x_fm_mean = np.array([x1_fm.mean(), x2_fm.mean(), x3_fm.mean()])
obj_fm = np.sum((x_fm_mean - x_target)**2)
print(f"{'Flow Matching':<20} {x_fm_mean[0]:<12.4f} {x_fm_mean[1]:<12.4f} {x_fm_mean[2]:<12.4f} {obj_fm:<12.6f}")

print()
print("Constraint satisfaction:")
print(f"  Scipy RON:  {rons @ x_scipy:.4f}, Sum: {x_scipy.sum():.6f}")
print(f"  GMM RON:    {rons @ x_gmm_mean:.4f}, Sum: {x_gmm_mean.sum():.6f}")
print(f"  FM RON:     {rons @ x_fm_mean:.4f}, Sum: {x_fm_mean.sum():.6f}")

### Summary: Blending Optimization

Both GMM and Flow Matching solve the equality-constrained quadratic programming problem by learning the joint distribution of decision variables, Lagrange multipliers, and their gradients, then conditioning on the KKT conditions (all partial derivatives = 0).

**Key observations:**
- The quadratic objective ensures an interior solution exists (no bound constraints active)
- The generative approach naturally handles the equality constraints through the Lagrangian formulation
- Both methods produce distributions centered near the scipy solution
- The constraints (RON = 87, sum = 1) are approximately satisfied in expectation